### This notebook compute "P4. Change in mean annual temperature" indicator for the 27 basins of IKI Project

**Created:** Sophia Bakar (sbakar@rti.org) 

**Project #:** 0219481  

**Last modified:** 1/2/2026

**Status:** complete for baseline and future scenario; needs modification to account for additional future scenarios or other met sources

**QA Status:** reviewed by  

**Original Script Stored at:** Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\Peligro\Scripts_Peligro
 
**Objective:**   

**Compatibility:** 

**Packages:** numpy, pandas, geopandas, sqlite3

**Further documentation:**  
 
**Inputs:**   subbasins shapefile, modeling groups databases (met table)

**Outputs:** 
 
**Assumptions:** 
 
**Future work:** 
 
**Notes:** 

In [1]:
import numpy as np
import pandas as pd
import sqlite3
import geopandas as gpd

In [2]:
#user = 'jmayo'
#user= 'sgilson'
user = 'sbakar'
#db_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db'
db_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db"

In [3]:
IndID = 104 #Indicator ID (Peligro = 1 + 0X where X is the Peligro Indicator number) 
met_sources = {
    "baseline": 2,  # PISCO / historical
    "future":   5   # CMIP6 85
} 

# get scenarios from database 
conn = sqlite3.connect(db_path)

scenarios_df = pd.read_sql_query(
    """
    SELECT ScnID, ScnName
    FROM ScnMod
    """,
    conn
)

conn.close()

# For now: apply the same values to all scenarios
scenario_ids = scenarios_df['ScnID'].tolist()

In [4]:
#subbasins_shapefile = fr"C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\GIS_WaterALLOC_General\Peru_AHD_with_districts.shp"
subbasins_shapefile = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\GIS_WaterALLOC_General\Peru_AHD_with_districts.shp"
subbasins_gdf = gpd.read_file(subbasins_shapefile).set_index('COMID').to_crs('WGS84')

In [5]:
temperature_change_all = []

start_year = 1981
end_year = 2020

In [6]:
# function to query the modeling databases for temperature and calculate mean annual temperature by COMID for a given met_source_id

def compute_mean_annual_temp(met_source_id):
    """Returns dataframe: comid, mean_annual_temp"""

    results = []

    for grupo in range(1, 13):

        #sqlite_path = f'C:/Users/{user}/Research Triangle Institute/IKI Peru Project - General/Interno/AI2b_Modelacion/Grupos_Modelacion/Grupo_{grupo}/BD/BD_Grupo_{grupo}.sqlite'
        sqlite_path = f"C:/Users/sbakar/OneDrive - Research Triangle Institute/IKI Peru Project - General/Interno/AI2b_Modelacion/Grupos_Modelacion/Grupo_{grupo}/BD/BD_Grupo_{grupo}.sqlite"
        conn = sqlite3.connect(sqlite_path)

        query = f"""
        SELECT comid, avg_temp_c, measured_date
        FROM catchment_met_observations
        WHERE met_source_id = {met_source_id};
        """

        met = pd.read_sql_query(query, conn)
        conn.close()

        if met.empty:
            continue

        met['measured_date'] = pd.to_datetime(
            met['measured_date'],
            format='%Y-%m-%d %H:%M:%S %z UTC',
            errors='coerce'
        )
        met = met.dropna(subset=['measured_date'])

        met['year'] = met['measured_date'].dt.year
        met['month'] = met['measured_date'].dt.month

        met = met[
            (met['year'] >= start_year) &
            (met['year'] <= end_year)
        ]

        # Monthly means
        monthly = (
            met.groupby(['comid', 'year', 'month'], as_index=False)['avg_temp_c']
            .mean()
        )

        # Annual means
        annual = (
            monthly.groupby(['comid', 'year'], as_index=False)['avg_temp_c']
            .mean()
            .rename(columns={'avg_temp_c': 'mean_annual_temp'})
        )

        results.append(annual)

    df = pd.concat(results, ignore_index=True)

    return (
        df.groupby('comid', as_index=False)['mean_annual_temp']
          .mean()
    )


In [8]:
# compute mean annual temperature for baseline and future scenarios 
## figure out a way to generalize this for multiple scenarios later

baseline_temp = compute_mean_annual_temp(met_sources['baseline'])
baseline_temp = baseline_temp.rename(
    columns={'mean_annual_temp': 'baseline_mean_temp'}
)

future_temp = compute_mean_annual_temp(met_sources['future'])
future_temp = future_temp.rename(
    columns={'mean_annual_temp': 'future_mean_temp'}
)

In [9]:
# compute the change in mean annual temperature between the baseline and future period 
## change to account for multiple future scenarios...

temperature_change = baseline_temp.merge(
    future_temp,
    on='comid',
    how='inner'
)

temperature_change['delta_temp'] = (
    temperature_change['future_mean_temp']
    - temperature_change['baseline_mean_temp']
)

In [11]:
# classify delta_temp into hazard levels

def classify_peligro(delta):
    if delta > 2.0:
        return "Muy Alto"
    elif 1.5 <= delta <= 2.0:
        return "Alto"
    elif 1.0 <= delta < 1.5:
        return "Medio"
    else:
        return "Bajo"

value_map = {
    "Muy Alto": 4,
    "Alto": 3,
    "Medio": 2,
    "Bajo": 1
}

temperature_change['Nivel_de_Peligro'] = (
    temperature_change['delta_temp'].apply(classify_peligro)
)
temperature_change['Value'] = (
    temperature_change['Nivel_de_Peligro'].map(value_map)
)

In [12]:
rows_to_insert = []

for ScnID in scenario_ids:
    for _, row in temperature_change.iterrows():
        rows_to_insert.append((
            ScnID,
            IndID,
            int(row['comid']),
            int(row['Value'])
        ))


In [13]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

insert_query = """
INSERT OR REPLACE INTO IndValues_Dyn (ScnID, IndID, COMID, Value)
VALUES (?, ?, ?, ?);
"""

In [15]:
# check that min and max values match the expected range based on the Indicators Table 
indicator_limits = pd.read_sql_query(
    """
    SELECT IndID, Min, Max
    FROM Indicators
    WHERE IndID = ?
    """,
    conn,
    params=(IndID,)
)

if indicator_limits.empty:
    raise ValueError(f"No entry found in Indicators table for IndID = {IndID}")

ind_min = indicator_limits.loc[0, 'Min']
ind_max = indicator_limits.loc[0, 'Max']

print(f"Indicator {IndID}  Min: {ind_min}, Max: {ind_max}")

value_stats = (
    temperature_change['Value']
    .agg(['min', 'max', 'count'])
    .reset_index()
)

print("\n=== Values to be inserted (by scenario) ===")
print(value_stats)

# Check for duplicates in the input dataframe before insert
df_check = pd.DataFrame(rows_to_insert, columns=['ScnID', 'IndID', 'COMID', 'Value'])
duplicates = df_check.duplicated(subset=['ScnID', 'IndID', 'COMID'])
print("Duplicates in rows_to_insert:", df_check[duplicates])

Indicator 104  Min: 1, Max: 4

=== Values to be inserted (by scenario) ===
   index  Value
0    min      1
1    max      4
2  count   3054
Duplicates in rows_to_insert: Empty DataFrame
Columns: [ScnID, IndID, COMID, Value]
Index: []


In [16]:
#Execute insert to SQLite Database
cursor.executemany(insert_query, rows_to_insert)
conn.commit()
conn.close()